In [ ]:
import os
import pandas as pd

from sklearn.metrics import (
    normalized_mutual_info_score,
    v_measure_score
)

# --------------------------
# Paths
# --------------------------
MANUAL_FILE = "protocol_manual_classification_updated.csv"
CLUSTER_BASE = "clusters"

# --------------------------
# Load manual labels
# --------------------------
manual_df = pd.read_csv(MANUAL_FILE)

manual_df["Word"] = manual_df["Word"].astype(str).str.strip().str.lower()
manual_df["REMARK"] = manual_df["REMARK"].astype(str).str.strip().str.lower()

manual_df["manual_cluster"] = manual_df["REMARK"].astype("category").cat.codes

# --------------------------
# Load predicted clusters
# --------------------------
def load_cluster_folder(folder_path):
    pred_map = {}

    for file in os.listdir(folder_path):
        if file.endswith(".csv") and "cluster_" in file.lower():
            cluster_id = int(file.lower().replace("cluster_", "").replace(".csv", ""))

            df = pd.read_csv(os.path.join(folder_path, file))
            df["Word"] = df["Word"].astype(str).str.strip().str.lower()

            for word in df["Word"]:
                pred_map[word] = cluster_id

    return pred_map


# --------------------------
# Evaluate each model
# --------------------------
results = []

for model_folder in os.listdir(CLUSTER_BASE):
    folder_path = os.path.join(CLUSTER_BASE, model_folder)

    if not os.path.isdir(folder_path):
        continue

    pred_map = load_cluster_folder(folder_path)

    temp = manual_df.copy()
    temp["predicted_cluster"] = temp["Word"].map(pred_map)

    aligned = temp.dropna(subset=["predicted_cluster"]).copy()
    aligned["predicted_cluster"] = aligned["predicted_cluster"].astype(int)

    y_true = aligned["manual_cluster"]
    y_pred = aligned["predicted_cluster"]

    nmi = normalized_mutual_info_score(
        y_true,
        y_pred,
        average_method="geometric"
    )

    vscore = v_measure_score(y_true, y_pred)

    results.append({
        "Model": model_folder,
        "NMI": nmi,
        "V_measure": vscore,
        "Matched_Words": len(aligned),
        "Unmatched_Words": len(temp) - len(aligned)
    })


# --------------------------
# Save results
# --------------------------
results_df = pd.DataFrame(results).sort_values("NMI", ascending=False)

print("\nEvaluation Results:\n")
print(results_df)

results_df.to_csv("all_model_nmi_vmeasure.csv", index=False)

print("\nSaved to: all_model_nmi_vmeasure.csv")

KeyError: 'ARI'